# Pneumonia Detection in Pediatric Chest X-rays
## Reproducing Kundu et al. (2021) and testing one change: CLAHE pre-processing

**Team:** Aman Kumar (230065) · Asmit (230063) · Shashwat (230127) — Newton School of Technology, Rishihood University

**Base paper:** Kundu R., Das R., Geem Z.W., Han G.-T., Sarkar R. (2021). *Pneumonia detection in chest X-ray images using an ensemble of deep learning models.* PLOS ONE 16(9): e0256630. [doi:10.1371/journal.pone.0256630](https://doi.org/10.1371/journal.pone.0256630) · [official code](https://github.com/Rohit-Kundu/Ensemble-Pneumonia-Detection)

**Dataset:** Kermany pediatric chest X-ray dataset (5,856 images: 1,583 Normal, 4,273 Pneumonia), via Kaggle `andrewmvd/pediatric-pneumonia-chest-xray`.

| Milestone | Where in this notebook |
|---|---|
| **M2** Baseline reproduction | §3 method · §6.1 training · §7 results and comparison with the paper |
| **M3** Hypothesis + prediction, submitted before coding | `M3_PREDICTIONS` in §1 · checked in §8.1 |
| **M4** The one change (CLAHE) + ablations | §3 pre-processing · §6.2–6.3 training · §8 results |
| **M5** Failure analysis | §6.4 (optional run) · §9 analysis · §10 summary |

### How to run (Google Colab, T4 GPU)
1. **§1–§5** — setup, data, method, engine and checks. Minutes; nothing is trained.
2. **§6** trains only what is missing. Every model saves a checkpoint to Google Drive after each epoch, so a dropped Colab session resumes where it stopped, and finished folds are skipped. **The M2 baseline is already on Drive, so §6.1 finishes in seconds.**
3. **§7–§10** only read saved files and can run at any time. Tip: run §7 and §9 *before* starting §6.2, because the kernel is busy for hours while training runs.

*Training log:* the 15 baseline models were trained with the earlier notebook `Kundu_Stage3_Baseline_Reproduction_CHECKPOINTED.ipynb`, which is kept unchanged as the record of that run. This notebook reads its checkpoints and predictions from the same Drive folder. The training protocol is identical. The only engine differences: new runs seed each (fold, model) pair separately and restore the random-number state exactly on resume, while the original run used one global seed.

## 1. Setup and configuration

In [ ]:
import os, sys, re, gc, json, copy, time, random, hashlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image, ImageFile
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix)
from scipy import stats
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ---------- storage ----------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = "/content/drive/MyDrive/CV_Project/Kundu_Stage3_Checkpoints"   # same folder as the M2 run
    CACHE_ROOT = "/content/m4_cache"                                             # fast local disk, rebuilt if lost
else:
    OUTPUT_ROOT, CACHE_ROOT = "outputs", "cache"
M4_ROOT = os.path.join(OUTPUT_ROOT, "M4_CLAHE")
M5_ROOT = os.path.join(OUTPUT_ROOT, "M5_failure_analysis")
for d in (OUTPUT_ROOT, M4_ROOT, M5_ROOT):
    os.makedirs(d, exist_ok=True)

# ---------- training protocol (Kundu et al. + disclosed choices), identical for every experiment ----------
SEED, N_FOLDS, EPOCHS, BATCH_SIZE = 42, 5, 30, 16
LEARNING_RATE, INTERNAL_VAL_FRACTION, IMAGE_SIZE, NUM_WORKERS = 1e-4, 0.10, 224, 2
MODELS = ["googlenet", "resnet18", "densenet121"]
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]                       # label 0, label 1
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# ---------- M4: the one change and its ablations ----------
CLAHE_TILE = (8, 8)
VARIANTS = {                     # name: (enhancement applied before the resize, CLAHE clip limit)
    "clahe_c2":  ("clahe", 2.0), # E1  - the M3 hypothesis
    "global_he": ("he",    None),# E2a - global equalisation: is the LOCAL part what matters?
    "clahe_c1":  ("clahe", 1.0), # E2b - weaker CLAHE
    "clahe_c4":  ("clahe", 4.0), # E2c - stronger CLAHE
    "identity":  ("none",  None),# E2d - no enhancement, new pipeline/seeds: re-training noise floor
}
E1_VARIANT, E1_MODELS = "clahe_c2", MODELS
E2_VARIANTS, E2_MODELS = ["global_he", "clahe_c1", "clahe_c4"], ["resnet18"]
RUN_NOISE_FLOOR = False          # True -> also train E2d
RUN_GROUPED_CV = False           # True -> §6.4 patient-grouped CV (ResNet-18)
DELETE_INTERMEDIATE_CKPTS = True # keep only *_final.pt per finished model (Drive space)

# ---------- M3 predictions (written BEFORE any M4 code). Edit to match the M3 submission exactly. ----------
# allowed: "increase", "decrease", "no decrease", "no increase"
M3_PREDICTIONS = {"recall": "increase", "auc_probability": "increase", "f1": "no decrease"}

# ---------- numbers reported by Kundu et al. on this dataset (5-fold mean ± SD, %) ----------
PAPER = {"accuracy": (98.81, 0.61), "precision": (98.82, 0.59), "recall": (98.80, 0.60),
         "f1": (98.79, 0.61), "auc": (98.35, 0.68)}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False

seed_everything(SEED)
print(f"torch {torch.__version__} | device {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))
print("Results folder:", OUTPUT_ROOT)

## 2. Data
The official Kaggle copy comes split into `train` and `test` folders. As in Kundu et al., the two are **pooled** and evaluated with 5-fold cross-validation. Filenames also give the pneumonia subtype (`person12_virus_44.jpeg`) and a patient ID, which §9 uses for the failure analysis.

In [ ]:
EXPECTED = {"total": 5856, "NORMAL": 1583, "PNEUMONIA": 4273}    # Kundu et al.

def locate_dataset():
    if os.environ.get("CV_DATASET_ROOT"):
        return Path(os.environ["CV_DATASET_ROOT"])
    try:
        import kagglehub
    except ImportError:
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", "kagglehub"], check=True)
        import kagglehub
    try:
        base = Path(kagglehub.dataset_download("andrewmvd/pediatric-pneumonia-chest-xray"))
    except Exception:                      # private download needs a token
        from getpass import getpass
        os.environ["KAGGLE_API_TOKEN"] = getpass("Kaggle API token: ")
        base = Path(kagglehub.dataset_download("andrewmvd/pediatric-pneumonia-chest-xray"))
    candidates = [base] + [t.parent for t in base.rglob("train")]
    return next(p for p in candidates if all((p / s / c).is_dir() for s in ("train", "test") for c in CLASS_NAMES))

def parse_name(path):
    """('person12', 'virus') for PNEUMONIA files, ('IM-0115', 'normal') for NORMAL files, else (None, 'unknown')."""
    name = os.path.basename(path)
    m = re.match(r"^(person\d+)_(bacteria|virus)_\d+", name, re.I)
    if m:
        return m.group(1).lower(), m.group(2).lower()
    m = re.match(r"^((?:NORMAL2-)?IM-\d+)-\d+", name, re.I)
    return (m.group(1).upper(), "normal") if m else (None, "unknown")

DATASET_ROOT = locate_dataset()
rows = []
for split in ["train", "test", "val"]:          # same order as the M2 run -> same row order -> same folds
    for label, cls in enumerate(CLASS_NAMES):
        folder = DATASET_ROOT / split / cls
        if folder.exists():
            rows += [{"path": str(p), "class": cls, "label": label, "source_split": split}
                     for p in folder.rglob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")]
df = pd.DataFrame(rows).drop_duplicates(subset=["path"])

parsed = df["path"].map(parse_name)
df["patient_id"], df["subtype"] = parsed.str[0], parsed.str[1]
# Conservative patient key: the same ID inside the same original partition = the same child (lower bound on leakage).
df["patient_key"] = np.where(df["patient_id"].notna(), df["source_split"] + "/" + df["patient_id"].fillna(""), None)

counts = df["class"].value_counts()
print(f"Dataset root: {DATASET_ROOT}")
print(f"Images: {len(df)} | NORMAL {counts.get('NORMAL', 0)} | PNEUMONIA {counts.get('PNEUMONIA', 0)} | "
      f"matches Kundu et al.: {len(df) == EXPECTED['total'] and all(counts.get(c, 0) == EXPECTED[c] for c in CLASS_NAMES)}")
print(f"Filenames parsed for patient/subtype: {df['patient_id'].notna().mean():.1%}")
display(pd.crosstab([df["class"], df["subtype"]], df["source_split"], margins=True))

## 3. Method
**Baseline (Kundu et al.):** resize to 224×224 → ImageNet normalisation. Three ImageNet-pretrained CNNs (GoogLeNet, ResNet-18, DenseNet-121), all layers trainable, each with a new 2-class head. Their probabilities are combined with Kundu's weights, `w_i = tanh(P_i) + tanh(R_i) + tanh(F1_i) + tanh(AUC_i)`, computed on the training part of the fold and normalised to sum to 1.

**M4 change:** greyscale at native resolution → **CLAHE** → the *same* resize → the same normalisation. Enhanced images are cached once as 224×224 lossless PNGs, so training reads exactly the tensors an on-the-fly pipeline would produce. §5 checks this.

**Metrics** use the same definitions as Kundu's code: binary precision, recall and F1 with Pneumonia as the positive class. AUC is reported two ways: `auc_hard_label` (Kundu computes `roc_auc_score` on predicted labels, which equals balanced accuracy) and `auc_probability` (the standard, threshold-free ROC-AUC).

In [ ]:
# ---------- pre-processing ----------
baseline_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
cached_transform = transforms.Compose([           # cached images are already 224x224
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

def enhance_gray(gray, kind, clip):
    if kind == "clahe":
        return cv2.createCLAHE(clipLimit=clip, tileGridSize=CLAHE_TILE).apply(gray)
    return cv2.equalizeHist(gray) if kind == "he" else gray

def preprocess_to_224(src, kind, clip):
    """Native-resolution enhancement, then the SAME resize as the baseline. Returns a greyscale PIL image."""
    gray = np.array(Image.open(src).convert("L"))
    return transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))(Image.fromarray(enhance_gray(gray, kind, clip)))

_FRAMES = {}
def variant_frame(variant):
    """df plus 'input_path' = the file the network reads (original JPEG for the baseline, cached PNG otherwise)."""
    if variant not in _FRAMES:
        frame = df.copy()
        if variant == "baseline":
            frame["input_path"] = frame["path"]
        else:
            kind, clip = VARIANTS[variant]
            out = Path(CACHE_ROOT) / variant
            out.mkdir(parents=True, exist_ok=True)
            inputs, built = [], 0
            for src in frame["path"]:
                dst = out / (hashlib.sha1(src.encode()).hexdigest()[:20] + ".png")
                if not dst.exists():
                    preprocess_to_224(src, kind, clip).save(dst)
                    built += 1
                inputs.append(str(dst))
            frame["input_path"] = inputs
            print(f"[CACHE] {variant}: built {built}, reused {len(frame) - built}")
        _FRAMES[variant] = frame
    return _FRAMES[variant]

class XrayDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths, self.labels, self.transform = frame["input_path"].tolist(), frame["label"].tolist(), transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.transform(Image.open(self.paths[i]).convert("RGB")), self.labels[i]

def make_loader(frame, variant, shuffle):
    tf = baseline_transform if variant == "baseline" else cached_transform
    return DataLoader(XrayDataset(frame, tf), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

# ---------- models ----------
def build_model(name):
    if name == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, 2)
    elif name == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        model.classifier = nn.Linear(model.classifier.in_features, 2)
    elif name == "googlenet":   # torchvision needs aux_logits=True with pretrained weights; only the main output is trained
        model = models.googlenet(weights=models.GoogLeNet_Weights.DEFAULT, aux_logits=True)
        model.fc = nn.Linear(model.fc.in_features, 2)
        model.aux1.fc2 = nn.Linear(model.aux1.fc2.in_features, 2)
        model.aux2.fc2 = nn.Linear(model.aux2.fc2.in_features, 2)
    else:
        raise ValueError(name)
    for p in model.parameters():          # Kundu's best setting: all layers trainable
        p.requires_grad = True
    return model.to(DEVICE)

def extract_logits(output):
    return output.logits if hasattr(output, "logits") else output

def run_epoch(model, loader, criterion, optimizer=None):
    """One pass. Trains if an optimizer is given. Returns loss, accuracy, labels, class probabilities."""
    model.train(optimizer is not None)
    total, labels_all, probs_all = 0.0, [], []
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(optimizer is not None):
            logits = extract_logits(model(x))
            loss = criterion(logits, y)
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
        total += loss.item() * len(y)
        labels_all.append(y.cpu().numpy())
        probs_all.append(torch.softmax(logits, 1).detach().cpu().numpy())
    y_all, p_all = np.concatenate(labels_all), np.concatenate(probs_all)
    return total / len(y_all), float((p_all.argmax(1) == y_all).mean()), y_all, p_all

# ---------- metrics and Kundu ensemble ----------
def metrics(y, p_pneumonia):
    pred = (p_pneumonia > 0.5).astype(int)                 # == argmax over [P(normal), P(pneumonia)]
    rec_n = recall_score(y, pred, pos_label=0, zero_division=0)
    rec_p = recall_score(y, pred, pos_label=1, zero_division=0)
    return {"accuracy": accuracy_score(y, pred),
            "precision": precision_score(y, pred, zero_division=0),
            "recall": rec_p,
            "f1": f1_score(y, pred, zero_division=0),
            "auc_probability": roc_auc_score(y, p_pneumonia),
            "auc_hard_label": (rec_n + rec_p) / 2,
            "Normal_recall": rec_n, "Pneumonia_recall": rec_p}

def kundu_ensemble(test_probs, train_probs, y_train):
    m = [metrics(y_train, p[:, 1]) for p in train_probs]
    raw = np.array([sum(np.tanh(x[k]) for k in ("precision", "recall", "f1", "auc_probability")) for x in m])
    w = raw / raw.sum()
    return sum(wi * p for wi, p in zip(w, test_probs)), w

print("Method ready.")

## 4. Cross-validation folds and the checkpointed training engine
`run_fold(fold, variant, models)` trains each model, or resumes or skips it. It then predicts the held-out fold, builds the Kundu ensemble and writes `foldK_results.csv`, `foldK_predictions.csv` and, last of all, `foldK_DONE.json`. Folds come from `StratifiedKFold(5, shuffle=True, random_state=42)`. If the saved M2 predictions exist, their fold assignment is used, so every comparison with the baseline is on exactly the same held-out images.

In [ ]:
def out_dir_for(variant):
    return OUTPUT_ROOT if variant == "baseline" else os.path.join(M4_ROOT, variant)

def load_predictions(out_dir):
    """Held-out predictions of all COMPLETED folds in a results folder ('path' = original image)."""
    done = [k for k in range(1, N_FOLDS + 1) if os.path.exists(os.path.join(out_dir, f"fold{k}_DONE.json"))]
    if not done:
        return None
    preds = pd.concat([pd.read_csv(os.path.join(out_dir, f"fold{k}_predictions.csv")) for k in done], ignore_index=True)
    return preds.rename(columns={"orig_path": "path"})

def fold_map_from(splitter, frame, groups=None):
    return {p: k for k, (_, te) in enumerate(splitter.split(frame["path"], frame["label"], groups), start=1)
            for p in frame["path"].iloc[te]}

FOLD_OF_PATH = fold_map_from(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED), df)
_saved = load_predictions(OUTPUT_ROOT)
if _saved is not None and len(_saved) == len(df):
    _saved_map = dict(zip(_saved["path"], _saved["fold"]))
    if _saved_map != FOLD_OF_PATH:
        print("NOTE: today's split differs from the saved M2 folds -> using the saved folds (keeps comparisons paired).")
        FOLD_OF_PATH = _saved_map

def fold_frames(fold_idx, frame, fold_map):
    is_test = (frame["path"].map(fold_map) == fold_idx).values
    trainval, test = frame[~is_test].reset_index(drop=True), frame[is_test].reset_index(drop=True)
    train, val = train_test_split(trainval, test_size=INTERNAL_VAL_FRACTION,
                                  stratify=trainval["label"], random_state=SEED + fold_idx)
    return trainval, train, val, test

def _rng_state():
    return {"torch": torch.get_rng_state(), "numpy": np.random.get_state(), "python": random.getstate(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}

def _set_rng_state(s):
    torch.set_rng_state(s["torch"]); np.random.set_state(s["numpy"]); random.setstate(s["python"])
    if s["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(s["cuda"])

def load_trained(model_name, fold_idx, out_dir=OUTPUT_ROOT):
    """Best-validation weights of a finished model (works for the M2 run's checkpoints too)."""
    model = build_model(model_name)
    ck = torch.load(os.path.join(out_dir, f"fold{fold_idx}_{model_name}_final.pt"), map_location="cpu", weights_only=False)
    model.load_state_dict(ck["best_state_dict"])
    return model

def train_model(model_name, variant, fold_idx, train, val, out_dir):
    """Adam 1e-4, cross-entropy, ReduceLROnPlateau(0.1, patience 3) on val loss, keep best-val-loss weights.
    Checkpoint after every epoch -> exact resume after a disconnect."""
    f = lambda kind: os.path.join(out_dir, f"fold{fold_idx}_{model_name}_{kind}")
    if os.path.exists(f("final.pt")):
        return load_trained(model_name, fold_idx, out_dir)

    seed_everything(SEED + 1000 * fold_idx + 10 * MODELS.index(model_name))  # same seed in every variant
    model = build_model(model_name)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=3)
    train_loader, val_loader = make_loader(train, variant, True), make_loader(val, variant, False)
    best_loss, best_state, history, start = float("inf"), copy.deepcopy(model.state_dict()), [], 0

    if os.path.exists(f("latest.pt")):
        ck = torch.load(f("latest.pt"), map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model_state_dict"])
        optimizer.load_state_dict(ck["optimizer_state_dict"])
        scheduler.load_state_dict(ck["scheduler_state_dict"])
        start, best_loss, best_state, history = ck["epoch"], ck["best_val_loss"], ck["best_state_dict"], ck["history"]
        if "rng" in ck:
            _set_rng_state(ck["rng"])
        print(f"[RESUME] {variant} {model_name} fold {fold_idx} from epoch {start + 1}")

    for epoch in range(start, EPOCHS):
        t0 = time.time()
        tr_loss, tr_acc, _, _ = run_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_acc, _, _ = run_epoch(model, val_loader, criterion)
        scheduler.step(va_loss)
        lr = optimizer.param_groups[0]["lr"]
        history.append({"epoch": epoch + 1, "train_loss": tr_loss, "train_acc": tr_acc,
                        "val_loss": va_loss, "val_acc": va_acc, "lr": lr})
        if va_loss < best_loss:
            best_loss, best_state = va_loss, copy.deepcopy(model.state_dict())
        torch.save({"epoch": epoch + 1, "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(), "scheduler_state_dict": scheduler.state_dict(),
                    "best_val_loss": best_loss, "best_state_dict": best_state, "history": history,
                    "rng": _rng_state()}, f("latest.pt"))
        pd.DataFrame(history).to_csv(f("history.csv"), index=False)
        print(f"{variant:9s} {model_name:11s} f{fold_idx} | ep {epoch + 1:02d}/{EPOCHS} | train {tr_loss:.4f}/{tr_acc:.4f}"
              f" | val {va_loss:.4f}/{va_acc:.4f} | lr {lr:.0e} | {time.time() - t0:.0f}s")

    model.load_state_dict(best_state)
    torch.save({"best_state_dict": best_state, "best_val_loss": best_loss, "history": history}, f("final.pt"))
    if DELETE_INTERMEDIATE_CKPTS and os.path.exists(f("latest.pt")):
        os.remove(f("latest.pt"))
    return model

def run_fold(fold_idx, variant, models, out_dir=None, fold_map=None):
    out_dir, fold_map = out_dir or out_dir_for(variant), fold_map or FOLD_OF_PATH
    os.makedirs(out_dir, exist_ok=True)
    f = lambda name: os.path.join(out_dir, f"fold{fold_idx}_{name}")
    if os.path.exists(f("DONE.json")):
        print(f"[SKIP] {variant} fold {fold_idx}: already complete")
        return
    trainval, train, val, test = fold_frames(fold_idx, variant_frame(variant), fold_map)
    print(f"\n=== {variant} | fold {fold_idx} | train {len(train)} | val {len(val)} | test {len(test)} ===")

    criterion, p_train, p_test = nn.CrossEntropyLoss(), {}, {}
    for m in models:
        model = train_model(m, variant, fold_idx, train, val, out_dir)
        _, _, y_train, p_train[m] = run_epoch(model, make_loader(trainval, variant, False), criterion)
        _, _, y_test, p_test[m] = run_epoch(model, make_loader(test, variant, False), criterion)
        del model; gc.collect(); torch.cuda.empty_cache()

    preds = pd.DataFrame({"path": test["path"], "true_label": y_test, "fold": fold_idx,
                          **{f"{m}_pneumonia": p_test[m][:, 1] for m in models}})
    rows = [{"model": m, **metrics(y_test, p_test[m][:, 1])} for m in models]
    if len(models) > 1:
        ens, w = kundu_ensemble([p_test[m] for m in models], [p_train[m] for m in models], y_train)
        preds["ensemble_pneumonia"] = ens[:, 1]
        rows.insert(0, {"model": "ensemble", **metrics(y_test, ens[:, 1]),
                        **{f"weight_{m}": wi for m, wi in zip(models, w)}})
    pd.DataFrame(rows).assign(fold=fold_idx).to_csv(f("results.csv"), index=False)
    preds.to_csv(f("predictions.csv"), index=False)
    with open(f("DONE.json"), "w") as fh:     # written last = the fold is complete
        json.dump({"fold": fold_idx, "variant": variant, "models": models,
                   "completed_at": datetime.now().isoformat()}, fh)
    print(f"[DONE] {variant} fold {fold_idx}: " + ", ".join(f"{r['model']} acc {r['accuracy']:.4f}" for r in rows))

print("Engine ready. Folds:", pd.Series(FOLD_OF_PATH).value_counts().sort_index().to_dict())

## 5. Pre-flight checks (no training)

In [ ]:
# (a) The saved M2 folds are disjoint and cover every image exactly once.
base_pred = load_predictions(OUTPUT_ROOT)
if base_pred is not None:
    print(f"M2 baseline folds on Drive: {sorted(base_pred['fold'].unique())}")
    if base_pred["fold"].nunique() == N_FOLDS:
        assert len(base_pred) == base_pred["path"].nunique() == len(df), "saved folds overlap or miss images"
        assert set(base_pred["path"]) == set(df["path"]), "dataset paths differ from the M2 run"
        print("Held-out folds: disjoint, cover all images, identical for every experiment below.")

# (b) Same row order as the M2 run -> the internal 10% validation split is identical too.
today = fold_map_from(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED), df)
print("Internal validation split identical to M2:", today == FOLD_OF_PATH)

# (c) With NO enhancement, the cached pipeline must reproduce the baseline tensors.
sample = df.sample(min(40, len(df)), random_state=SEED)["path"]
diff = max(float((baseline_transform(Image.open(p).convert("RGB"))
                  - cached_transform(preprocess_to_224(p, "none", None).convert("RGB"))).abs().max()) for p in sample)
print(f"Cached vs baseline pipeline, max |difference|: {diff:.2e}  (0 for greyscale source images)")

# (d) What every variant looks like to the network.
show = pd.concat([df[df.label == 0].sample(1, random_state=1), df[df.label == 1].sample(2, random_state=1)])
names = ["baseline"] + list(VARIANTS)
fig, axes = plt.subplots(len(show), len(names), figsize=(2.5 * len(names), 2.7 * len(show)), squeeze=False)
for r, (_, row) in enumerate(show.iterrows()):
    for c, name in enumerate(names):
        kind, clip = ("none", None) if name == "baseline" else VARIANTS[name]
        axes[r, c].imshow(preprocess_to_224(row["path"], kind, clip), cmap="gray", vmin=0, vmax=255)
        axes[r, c].set_title(f"{name}\n{row['class']}" if r == 0 else row["class"], fontsize=9)
        axes[r, c].axis("off")
plt.tight_layout(); plt.savefig(os.path.join(M4_ROOT, "variant_examples.png"), dpi=150); plt.show()

## 6. Experiments
Every cell here is **safe to re-run**: finished folds print `[SKIP]`, and an interrupted model resumes from its last epoch. After a Colab disconnect, run §1–§4 again, then the cell that was running.

### 6.1 M2 — baseline reproduction (3 backbones + Kundu ensemble, 5 folds)
Already trained, so this cell only confirms all five folds are on Drive.

In [ ]:
for k in range(1, N_FOLDS + 1):
    run_fold(k, "baseline", MODELS)

### 6.2 M4 · E1 — CLAHE (clip limit 2.0) on all three backbones + ensemble
The main test of the M3 hypothesis. CLAHE is the only difference from §6.1. Time the first epoch and multiply by 450 (5 folds × 3 models × 30 epochs) to estimate the total.

In [ ]:
for k in range(1, N_FOLDS + 1):
    run_fold(k, E1_VARIANT, E1_MODELS)

### 6.3 M4 · E2 — ablations (ResNet-18 only, in priority order)
Global HE asks whether it is *local* contrast that matters. Clip limits 1 and 4 ask whether the effect depends on strength. `identity` (optional) is the same new pipeline and seeds with no enhancement: the noise floor.

In [ ]:
for v in E2_VARIANTS + (["identity"] if RUN_NOISE_FLOOR else []):
    for k in range(1, N_FOLDS + 1):
        run_fold(k, v, E2_MODELS)

### 6.4 M5 · optional — patient-grouped cross-validation (ResNet-18)
Kundu's protocol (and ours) splits by **image**, so one child's X-rays can sit on both sides of a split. This run keeps each patient in a single fold, which measures how much that leakage inflates the score. It uses the no-enhancement pipeline, so compare it with `identity` (§6.3), or with the baseline ResNet-18 if `identity` wasn't run.

In [ ]:
GROUPED_DIR = os.path.join(M5_ROOT, "grouped_cv", "identity")
GROUPED_FOLD_OF_PATH = fold_map_from(StratifiedGroupKFold(N_FOLDS, shuffle=True, random_state=SEED), df,
                                     groups=df["patient_key"].fillna("img:" + df["path"]))
assert df.assign(f=df["path"].map(GROUPED_FOLD_OF_PATH)).dropna(subset=["patient_key"]) \
         .groupby("patient_key")["f"].nunique().max() == 1, "a patient spans several folds"
print("Grouped folds (test images):", pd.Series(GROUPED_FOLD_OF_PATH).value_counts().sort_index().to_dict())
if RUN_GROUPED_CV:
    for k in range(1, N_FOLDS + 1):
        run_fold(k, "identity", ["resnet18"], out_dir=GROUPED_DIR, fold_map=GROUPED_FOLD_OF_PATH)

## 7. M2 results — baseline reproduction
All tables are recomputed from the saved held-out predictions with the single `metrics()` function in §3, so every experiment uses identical definitions. Values are five-fold **mean ± SD in %**.

In [ ]:
# ---------- load every finished experiment ----------
RESULT_DIRS = {"baseline": OUTPUT_ROOT, **{v: os.path.join(M4_ROOT, v) for v in VARIANTS},
               "grouped_cv": os.path.join(M5_ROOT, "grouped_cv", "identity")}
PRED = {name: p for name, d in RESULT_DIRS.items() if (p := load_predictions(d)) is not None}
METRICS = ["accuracy", "precision", "recall", "f1", "auc_probability", "auc_hard_label",
           "Normal_recall", "Pneumonia_recall"]
PER_FOLD = pd.DataFrame([
    {"experiment": name, "model": col[:-len("_pneumonia")], "fold": k, **metrics(g["true_label"].values, g[col].values)}
    for name, p in PRED.items() for col in p.columns if col.endswith("_pneumonia") for k, g in p.groupby("fold")])
PER_FOLD.to_csv(os.path.join(OUTPUT_ROOT, "per_fold_metrics_all_experiments.csv"), index=False)
print("Finished experiments (folds):", {n: sorted(p["fold"].unique().tolist()) for n, p in PRED.items()})

def fmt(s):
    return f"{s.mean() * 100:.2f} ± {s.std() * 100:.2f}"

def summary(exp, models):
    t = PER_FOLD[PER_FOLD.experiment == exp].groupby("model")[METRICS].agg(fmt)
    return t.loc[[m for m in models if m in t.index]]

def paired(exp, model, metric, ref="baseline"):
    """Variant minus reference on the SAME held-out folds (percentage points) + paired t-test."""
    sel = lambda e: PER_FOLD[(PER_FOLD.experiment == e) & (PER_FOLD.model == model)].set_index("fold")[metric]
    a, b = sel(ref), sel(exp)
    folds = a.index.intersection(b.index)
    d = (b[folds] - a[folds]) * 100
    p = stats.ttest_rel(b[folds], a[folds]).pvalue if len(folds) > 1 and d.std() > 0 else np.nan
    return {"reference_mean": a[folds].mean() * 100, "variant_mean": b[folds].mean() * 100,
            "delta_pp": d.mean(), "delta_sd": d.std(), "p_paired_t": p, "n_folds": len(folds)}

def mcnemar(exp, model, ref="baseline"):
    """Exact McNemar test on the pooled held-out images both experiments predicted."""
    col = f"{model}_pneumonia"
    m = PRED[ref][["path", "true_label", col]].merge(PRED[exp][["path", col]], on="path", suffixes=("_ref", "_var"))
    ok_r = ((m[col + "_ref"] > 0.5).astype(int) == m.true_label).values
    ok_v = ((m[col + "_var"] > 0.5).astype(int) == m.true_label).values
    b, c = int((ok_r & ~ok_v).sum()), int((~ok_r & ok_v).sum())
    return {"images": len(m), "only_reference_right": b, "only_variant_right": c,
            "p_mcnemar": stats.binomtest(min(b, c), b + c, 0.5).pvalue if b + c else 1.0}

# ---------- 7.1 baseline table ----------
print("M2 baseline - five-fold mean ± SD (%)")
display(summary("baseline", ["ensemble"] + MODELS))

# ---------- 7.2 pooled confusion matrix + ROC of the ensemble ----------
bp = PRED["baseline"]
y, p = bp["true_label"].values, bp["ensemble_pneumonia"].values
cm = confusion_matrix(y, (p > 0.5).astype(int), labels=[0, 1])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4))
ax1.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax1.text(j, i, f"{['TN', 'FP', 'FN', 'TP'][2 * i + j]}\n{cm[i, j]:,}", ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=12)
ax1.set(xticks=[0, 1], yticks=[0, 1], xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        xlabel="Predicted", ylabel="True", title="Kundu ensemble - pooled out-of-fold")
fpr, tpr, _ = roc_curve(y, p)
ax2.plot(fpr, tpr, label=f"ensemble, AUC = {roc_auc_score(y, p):.4f}")
ax2.plot([0, 1], [0, 1], "--", color="grey")
ax2.set(xlabel="False positive rate", ylabel="True positive rate", title="ROC (pooled out-of-fold)")
ax2.legend(loc="lower right")
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_ROOT, "baseline_confusion_roc.png"), dpi=200); plt.show()

### 7.3 Comparison with Kundu et al. using the same metric definitions
Kundu's `ensemble.py` computes binary precision, recall and F1, which match ours. Its AUC, however, is `roc_auc_score(labels, preds)` on **hard predicted labels**, which equals balanced accuracy. So the like-for-like comparison uses our `auc_hard_label`. Our standard probability ROC-AUC is listed separately; it is a different quantity and is not comparable with the paper's figure.

In [ ]:
ens = PER_FOLD.query("experiment == 'baseline' and model == 'ensemble'")
rows = []
for label, key, ours in [("Accuracy", "accuracy", "accuracy"), ("Precision", "precision", "precision"),
                         ("Recall", "recall", "recall"), ("F1", "f1", "f1"),
                         ("AUC (hard labels - Kundu's definition)", "auc", "auc_hard_label")]:
    pm, ps = PAPER[key]
    rows.append({"Metric": label, "Kundu et al.": f"{pm:.2f} ± {ps:.2f}", "Ours": fmt(ens[ours]),
                 "Gap (pp)": f"{ens[ours].mean() * 100 - pm:+.2f}"})
rows.append({"Metric": "ROC-AUC (probabilities - standard)", "Kundu et al.": "not reported",
             "Ours": fmt(ens["auc_probability"]), "Gap (pp)": "-"})
comparison = pd.DataFrame(rows)
comparison.to_csv(os.path.join(OUTPUT_ROOT, "paper_vs_reproduction_matched.csv"), index=False)
display(comparison)

## 8. M4 results — CLAHE
- **Δ** = CLAHE − baseline on the same held-out fold, in percentage points, mean ± SD over folds, with a paired t-test. With 5 folds this test has low power.
- **McNemar's exact test** compares, image by image, what only the baseline gets right with what only CLAHE gets right.
- A Δ smaller than its SD, or than the `identity` noise floor, should be called *no measurable effect*.

In [ ]:
# ---------- 8.1 E1: ensemble, every backbone, and the M3 prediction ----------
if E1_VARIANT not in PRED:
    print("E1 has no finished folds yet - run §6.2.")
else:
    e1 = pd.DataFrame([{"model": m, "metric": met, **paired(E1_VARIANT, m, met)}
                       for m in ["ensemble"] + E1_MODELS for met in METRICS])
    e1.to_csv(os.path.join(M4_ROOT, "E1_paired_deltas.csv"), index=False)

    t = e1[e1.model == "ensemble"].assign(**{"delta (pp)": lambda d: [f"{a:+.2f} ± {b:.2f}" for a, b in zip(d.delta_pp, d.delta_sd)]})
    print("E1 - Kundu ensemble, baseline vs CLAHE (clip 2.0)")
    display(t[["metric", "reference_mean", "variant_mean", "delta (pp)", "p_paired_t", "n_folds"]].round(3))
    print("Δ per backbone (pp) - does CLAHE help every backbone?")
    display(e1.pivot(index="model", columns="metric", values="delta_pp").loc[["ensemble"] + E1_MODELS, METRICS].round(2))
    print("McNemar (pooled held-out images)")
    display(pd.DataFrame([{"model": m, **mcnemar(E1_VARIANT, m)} for m in ["ensemble"] + E1_MODELS]).round(4))

    ok = {"increase": lambda d: d > 0, "decrease": lambda d: d < 0,
          "no decrease": lambda d: d >= 0, "no increase": lambda d: d <= 0}
    check = []
    for metric, predicted in M3_PREDICTIONS.items():
        r = paired(E1_VARIANT, "ensemble", metric)
        clear = abs(r["delta_pp"]) >= r["delta_sd"] and r["p_paired_t"] < 0.05
        check.append({"metric": metric, "M3 prediction": predicted,
                      "observed Δ (pp)": f"{r['delta_pp']:+.2f} ± {r['delta_sd']:.2f}",
                      "direction": "consistent" if ok[predicted](r["delta_pp"]) else "contradicted",
                      "beyond fold noise": "yes" if clear else "no - inconclusive", "p": r["p_paired_t"]})
    print("Was the M3 prediction right?")
    display(pd.DataFrame(check).round(4))

In [ ]:
# ---------- 8.2 E2: ablation on ResNet-18 ----------
abl = [v for v in VARIANTS if v in PRED and f"resnet18_pneumonia" in PRED[v].columns]
if not abl:
    print("No ResNet-18 variant results yet - run §6.2/§6.3.")
else:
    a = pd.DataFrame([{"variant": v, "metric": met, **paired(v, "resnet18", met)} for v in abl for met in METRICS])
    a.to_csv(os.path.join(M4_ROOT, "E2_ablation_resnet18.csv"), index=False)
    a["cell"] = [f"{d:+.2f} ± {s:.2f}" for d, s in zip(a.delta_pp, a.delta_sd)]
    print("ResNet-18: Δ vs baseline (pp)")
    display(a.pivot(index="variant", columns="metric", values="cell").loc[abl, METRICS])
    display(pd.DataFrame([{"variant": v, **mcnemar(v, "resnet18")} for v in abl]).round(4))

    shown = ["accuracy", "auc_probability", "Normal_recall", "Pneumonia_recall"]
    base_r = PER_FOLD.query("experiment == 'baseline' and model == 'resnet18'").set_index("fold")
    fig, axes = plt.subplots(1, len(shown), figsize=(4.2 * len(shown), 3.8))
    for ax, met in zip(axes, shown):
        for i, v in enumerate(abl):
            var = PER_FOLD.query("experiment == @v and model == 'resnet18'").set_index("fold")
            d = (var[met] - base_r.loc[var.index, met]) * 100
            ax.scatter([i] * len(d), d, s=18, alpha=0.6)
            ax.errorbar(i, d.mean(), yerr=d.std() if len(d) > 1 else 0, fmt="o", color="black", capsize=4)
        ax.axhline(0, color="grey", ls="--", lw=1)
        ax.set_xticks(range(len(abl)), abl, rotation=30, ha="right")
        ax.set_title(f"Δ {met} (pp)")
    plt.suptitle("ResNet-18: change vs baseline per fold (dots), mean ± SD")
    plt.tight_layout(); plt.savefig(os.path.join(M4_ROOT, "E2_ablation_resnet18.png"), dpi=200); plt.show()

In [ ]:
# ---------- 8.3 What moved: images CLAHE fixed or broke (ensemble) ----------
if E1_VARIANT in PRED:
    mv = PRED["baseline"][["path", "true_label", "ensemble_pneumonia"]].merge(
        PRED[E1_VARIANT][["path", "ensemble_pneumonia"]], on="path", suffixes=("_base", "_clahe")) \
        .merge(df[["path", "class", "subtype"]], on="path")
    ok_b = (mv.ensemble_pneumonia_base > 0.5).astype(int) == mv.true_label
    ok_c = (mv.ensemble_pneumonia_clahe > 0.5).astype(int) == mv.true_label
    mv["change"] = np.select([ok_b & ~ok_c, ~ok_b & ok_c, ~ok_b & ~ok_c],
                             ["broken by CLAHE", "fixed by CLAHE", "wrong in both"], "right in both")
    mv.to_csv(os.path.join(M4_ROOT, "E1_ensemble_flips.csv"), index=False)
    display(pd.crosstab(mv["change"], [mv["class"], mv["subtype"]], margins=True))

    ex = pd.concat([mv[mv.change == "fixed by CLAHE"].head(3), mv[mv.change == "broken by CLAHE"].head(3)])
    if len(ex):
        fig, axes = plt.subplots(len(ex), 2, figsize=(6, 3 * len(ex)), squeeze=False)
        for r, (_, row) in enumerate(ex.iterrows()):
            for c, (kind, clip, pcol, title) in enumerate([("none", None, "ensemble_pneumonia_base", "baseline"),
                                                           (*VARIANTS[E1_VARIANT], "ensemble_pneumonia_clahe", "CLAHE")]):
                axes[r, c].imshow(preprocess_to_224(row.path, kind, clip), cmap="gray", vmin=0, vmax=255)
                axes[r, c].set_title(f"{title}  p(pneumonia)={row[pcol]:.2f}", fontsize=9)
                axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
            axes[r, 0].set_ylabel(f"{row['class']}\n{row.change}", fontsize=9)
        plt.tight_layout(); plt.savefig(os.path.join(M4_ROOT, "E1_flip_examples.png"), dpi=150); plt.show()

## 9. M5 — failure analysis
The baseline ensemble is wrong on fewer than 2% of images. This section asks **which** images fail, **why**, and **whether the headline number can be trusted**. Everything reads saved files, except Grad-CAM (§9.6), which runs the saved ResNet-18 checkpoints forward and backward.

In [ ]:
# ---------- 9.0 one table: every held-out image with predictions + metadata ----------
fa = PRED["baseline"].merge(df[["path", "class", "source_split", "subtype", "patient_id", "patient_key"]],
                            on="path", validate="one_to_one")
fa["p"] = fa["ensemble_pneumonia"]
fa["correct"] = (fa.p > 0.5).astype(int) == fa.true_label
fa["outcome"] = np.select([fa.correct & (fa.true_label == 1), fa.correct & (fa.true_label == 0),
                           ~fa.correct & (fa.true_label == 0)], ["TP", "TN", "FP"], "FN")
HAS_CLAHE = E1_VARIANT in PRED and PRED[E1_VARIANT]["fold"].nunique() == N_FOLDS
if HAS_CLAHE:
    fa = fa.merge(PRED[E1_VARIANT][["path", "ensemble_pneumonia"]].rename(columns={"ensemble_pneumonia": "p_clahe"}), on="path")
print("Baseline ensemble outcomes:", fa["outcome"].value_counts().to_dict(), "| CLAHE results included:", HAS_CLAHE)

# ---------- 9.1 the most confident errors ----------
cols = ["file", "fold", "subtype", "source_split", "p"] + [f"{m}_pneumonia" for m in MODELS]
fa["file"] = fa["path"].map(os.path.basename)
for kind, ascending in [("FN", True), ("FP", False)]:
    print(f"Most confident {kind}s (p = ensemble P(pneumonia)):")
    display(fa[fa.outcome == kind].sort_values("p", ascending=ascending).head(8)[cols].round(3))

In [ ]:
# ---------- 9.2 calibration, confidence and decision threshold ----------
def reliability(y, p, bins=10):
    idx = np.clip((p * bins).astype(int), 0, bins - 1)
    b = [(p[idx == i].mean(), y[idx == i].mean(), (idx == i).sum()) for i in range(bins) if (idx == i).any()]
    conf, frac, n = map(np.array, zip(*b))
    return conf, frac, float(np.sum(n * np.abs(conf - frac)) / n.sum())

y = fa["true_label"].values
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
for name, col in [("ensemble", "p")] + [(m, f"{m}_pneumonia") for m in MODELS]:
    conf, frac, ece = reliability(y, fa[col].values)
    axes[0].plot(conf, frac, "o-", ms=4, label=f"{name}: ECE {ece:.3f}, Brier {np.mean((fa[col] - y) ** 2):.4f}")
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set(xlabel="mean predicted P(pneumonia)", ylabel="observed fraction pneumonia", title="Reliability")
axes[0].legend(fontsize=7)
for kind, color in [("TN", "tab:blue"), ("TP", "tab:green"), ("FP", "tab:orange"), ("FN", "tab:red")]:
    axes[1].hist(fa.loc[fa.outcome == kind, "p"], bins=40, range=(0, 1), alpha=0.6, label=kind, color=color)
axes[1].set(yscale="log", xlabel="ensemble P(pneumonia)", ylabel="images (log)", title="Confidence by outcome")
axes[1].legend()
thr = np.linspace(0.01, 0.99, 197)
sens = np.array([(fa.p[y == 1] > t).mean() for t in thr])
spec = np.array([(fa.p[y == 0] <= t).mean() for t in thr])
axes[2].plot(thr, sens * 100, label="Pneumonia recall (sensitivity)")
axes[2].plot(thr, spec * 100, label="Normal recall (specificity)")
axes[2].axvline(0.5, color="grey", ls="--", lw=1)
axes[2].set(xlabel="decision threshold", ylabel="%", ylim=(80, 100.5), title="Threshold trade-off")
axes[2].legend()
plt.tight_layout(); plt.savefig(os.path.join(M5_ROOT, "F1_calibration_threshold.png"), dpi=200); plt.show()

ops = []
for target in [0.995, 0.999, None]:
    t = 0.5 if target is None else (thr[np.where(sens >= target)[0][-1]] if (sens >= target).any() else np.nan)
    ops.append({"operating point": "default 0.5" if target is None else f"sensitivity >= {target}",
                "threshold": round(t, 3), "sensitivity": (fa.p[y == 1] > t).mean(),
                "specificity": (fa.p[y == 0] <= t).mean(),
                "missed pneumonia": int(((fa.p <= t) & (y == 1)).sum()), "false alarms": int(((fa.p > t) & (y == 0)).sum())})
print("Operating points (picked on the same data -> descriptive, not a validated clinical threshold):")
display(pd.DataFrame(ops).round(4))

In [ ]:
# ---------- 9.3 who fails? pneumonia subtype and original partition ----------
def error_rates(frame, pcol, by):
    err = (frame[pcol] > 0.5).astype(int) != frame["true_label"]
    return frame.assign(err=err).groupby(by)["err"].agg(images="size", errors="sum", error_rate="mean")

rates = {"baseline": error_rates(fa, "p", ["class", "subtype"])}
if HAS_CLAHE:
    rates["CLAHE"] = error_rates(fa, "p_clahe", ["class", "subtype"])
print("Error rate by class/subtype (pneumonia errors = missed cases; normal errors = false alarms)")
display(pd.concat(rates, axis=1).round(4))
pn = fa[fa["class"] == "PNEUMONIA"]
if {"bacteria", "virus"} <= set(pn["subtype"]):
    tab = pd.crosstab(pn["subtype"], pn["correct"]).reindex(index=["bacteria", "virus"], columns=[False, True], fill_value=0)
    print(f"Missed pneumonia: bacterial {1 - pn[pn.subtype == 'bacteria'].correct.mean():.2%} vs "
          f"viral {1 - pn[pn.subtype == 'virus'].correct.mean():.2%} (Fisher exact p = {stats.fisher_exact(tab.values)[1]:.3g})")
print("\nError rate by ORIGINAL Kermany partition (pooled for 5-fold CV):")
display(error_rates(fa, "p", ["source_split", "class"]).round(4))

In [ ]:
# ---------- 9.4 ensemble behaviour: shared mistakes, oracle bound, does the weighting matter? ----------
wrong = pd.DataFrame({m: (fa[f"{m}_pneumonia"] > 0.5).astype(int) != fa.true_label for m in MODELS})
print("Backbones wrong per image vs ensemble outcome:")
display(pd.crosstab(wrong.sum(axis=1).rename("backbones wrong"), fa.correct.map({True: "ensemble right", False: "ensemble wrong"}), margins=True))
print("Error overlap between backbones (Jaccard; 1 = identical mistakes):")
display(pd.DataFrame([[(wrong[a] & wrong[b]).sum() / max((wrong[a] | wrong[b]).sum(), 1) for b in MODELS] for a in MODELS],
                     index=MODELS, columns=MODELS).round(3))
print(f"Oracle (right if ANY backbone is right): {(~wrong.all(axis=1)).mean():.2%} vs ensemble {fa.correct.mean():.2%}")

probs = fa[[f"{m}_pneumonia" for m in MODELS]]
combos = {**{m: probs[f"{m}_pneumonia"] for m in MODELS}, "plain average": probs.mean(axis=1),
          "majority vote": ((probs > 0.5).sum(axis=1) >= 2).astype(float), "Kundu weighted": fa.p}
rows = []
for name, score in combos.items():
    per = pd.DataFrame([metrics(g.true_label.values, score[g.index].values) for _, g in fa.groupby("fold")])
    rows.append({"method": name, **{c: fmt(per[c]) for c in ["accuracy", "f1", "auc_probability", "Normal_recall", "Pneumonia_recall"]}})
print("Five-fold mean ± SD (%) - single backbones vs ways of combining them:")
display(pd.DataFrame(rows))
w = pd.concat([pd.read_csv(os.path.join(OUTPUT_ROOT, f"fold{k}_results.csv")) for k in range(1, N_FOLDS + 1)])
print("Normalised Kundu weights per fold (all ~0.333 means the weighting is effectively a plain average):")
display(w[w.model == "ensemble"][[f"weight_{m}" for m in MODELS]].describe().loc[["mean", "min", "max"]].round(5))

In [ ]:
# ---------- 9.5 patient-leakage audit ----------
def sibling_in_train(frame, key):
    """True if another image of the same patient sits in a different fold (= in this image's training data)."""
    return frame[key].notna() & (frame.groupby(key)["fold"].transform("nunique") > 1)

fa["sibling_in_train"] = sibling_in_train(fa, "patient_key")           # lower bound
fa["sibling_in_train_upper"] = sibling_in_train(fa, "patient_id")      # upper bound (ignores original partition)
per_patient = fa.dropna(subset=["patient_key"]).groupby("patient_key").size()
print(f"Patients: {len(per_patient)} | with >1 image: {(per_patient > 1).sum()} | max images per patient: {per_patient.max()}")
print(f"Held-out images with the same child in training: {fa.sibling_in_train.mean():.1%} (lower bound) to "
      f"{fa.sibling_in_train_upper.mean():.1%} (upper bound)")
for cls in CLASS_NAMES:
    s = fa[fa["class"] == cls]
    if s.sibling_in_train.nunique() == 2:
        tab = pd.crosstab(s.sibling_in_train, s.correct).reindex(index=[False, True], columns=[False, True], fill_value=0)
        print(f"{cls}: error rate {1 - s[~s.sibling_in_train].correct.mean():.2%} without vs "
              f"{1 - s[s.sibling_in_train].correct.mean():.2%} with a sibling in training "
              f"(Fisher p = {stats.fisher_exact(tab.values)[1]:.3g})")

if "grouped_cv" in PRED:          # result of the optional §6.4 run
    ref = "identity" if "identity" in PRED else "baseline"
    a = PER_FOLD.query("experiment == @ref and model == 'resnet18'")
    g = PER_FOLD.query("experiment == 'grouped_cv' and model == 'resnet18'")
    print(f"\nResNet-18: image-level folds ({ref}) vs patient-grouped folds")
    display(pd.DataFrame([{"metric": c, "image-level": fmt(a[c]), "patient-grouped": fmt(g[c]),
                           "difference (pp)": f"{(g[c].mean() - a[c].mean()) * 100:+.2f}",
                           "Welch p": stats.ttest_ind(g[c], a[c], equal_var=False).pvalue}
                          for c in ["accuracy", "f1", "auc_probability", "Normal_recall", "Pneumonia_recall"]]).round(4))
fa.to_csv(os.path.join(M5_ROOT, "all_heldout_predictions_with_metadata.csv"), index=False)

In [ ]:
# ---------- 9.6 training dynamics of the baseline models ----------
fig, axes = plt.subplots(1, len(MODELS), figsize=(5.2 * len(MODELS), 4), sharey=True)
dyn = []
for ax, m in zip(axes, MODELS):
    for k in range(1, N_FOLDS + 1):
        h = pd.read_csv(os.path.join(OUTPUT_ROOT, f"fold{k}_{m}_history.csv"))
        ax.plot(h.epoch, h.train_loss, color="tab:blue", alpha=0.5, lw=1, label="train" if k == 1 else None)
        ax.plot(h.epoch, h.val_loss, color="tab:red", alpha=0.6, lw=1, label="validation" if k == 1 else None)
        tiny = h.loc[h.lr <= 1e-6, "epoch"]
        dyn.append({"model": m, "fold": k, "best_val_epoch": int(h.loc[h.val_loss.idxmin(), "epoch"]),
                    "first_epoch_lr<=1e-6": int(tiny.iloc[0]) if len(tiny) else np.nan,
                    "final_train_acc": h.train_acc.iloc[-1]})
    ax.set(title=m, xlabel="epoch", yscale="log")
    ax.legend()
axes[0].set_ylabel("cross-entropy (log)")
plt.suptitle("Baseline: train vs internal-validation loss, 5 folds")
plt.tight_layout(); plt.savefig(os.path.join(M5_ROOT, "F5_training_dynamics.png"), dpi=200); plt.show()
display(pd.DataFrame(dyn).groupby("model").agg(["mean", "min", "max"]).drop(columns="fold").round(3))

In [ ]:
# ---------- 9.7 Grad-CAM (ResNet-18): where does the model look? ----------
# Each image is explained by the ResNet-18 of the fold in which it was HELD OUT. Heat = evidence for PNEUMONIA.
BORDER = 0.15                     # outer band on each side ~ outside the lung field (labels, markers, edges)
inner = np.zeros((IMAGE_SIZE, IMAGE_SIZE), bool)
b = int(IMAGE_SIZE * BORDER); inner[b:-b, b:-b] = True

def gradcam(model, x):
    store = {}
    def hook(_, __, out):
        store["act"] = out
        out.register_hook(lambda g: store.__setitem__("grad", g))
    h = model.layer4.register_forward_hook(hook)
    with torch.enable_grad():
        model.zero_grad(set_to_none=True)
        extract_logits(model(x))[:, 1].sum().backward()
    h.remove()
    cam = torch.relu((store["grad"].mean((2, 3), keepdim=True) * store["act"]).sum(1, keepdim=True))
    cam = nn.functional.interpolate(cam, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False)[0, 0]
    cam = cam.detach().cpu().numpy()
    return cam / (cam.max() + 1e-8)

targets = pd.concat([fa[~fa.correct], fa[fa.correct].sample(min(300, int(fa.correct.sum())), random_state=SEED)])
cams = {}
for k, g in targets.groupby("fold"):
    model = load_trained("resnet18", k).eval()
    for path in g["path"]:
        cams[path] = gradcam(model, baseline_transform(Image.open(path).convert("RGB")).unsqueeze(0).to(DEVICE))
    del model; gc.collect()
targets["border_share"] = targets["path"].map(lambda p: float(cams[p][~inner].sum() / (cams[p].sum() + 1e-8)))
err, cor = targets.loc[~targets.correct, "border_share"], targets.loc[targets.correct, "border_share"]
p_mw = stats.mannwhitneyu(err, cor).pvalue if len(err) and len(cor) else np.nan
print(f"Grad-CAM mass in the outer {BORDER:.0%} border: errors median {err.median():.3f} vs correct {cor.median():.3f} "
      f"(Mann-Whitney p = {p_mw:.3g}; a uniform map gives {1 - inner.mean():.3f})")
display(targets.groupby("outcome")["border_share"].describe()[["count", "mean", "50%"]].round(3))

gallery = pd.concat([fa[fa.outcome == "FN"].sort_values("p").head(4), fa[fa.outcome == "FP"].sort_values("p", ascending=False).head(4),
                     targets[targets.outcome == "TP"].head(2), targets[targets.outcome == "TN"].head(2)])
gallery = gallery[gallery["path"].isin(cams)]
fig, axes = plt.subplots(int(np.ceil(len(gallery) / 4)), 4, figsize=(13.6, 3.6 * np.ceil(len(gallery) / 4)), squeeze=False)
for ax, (_, r) in zip(axes.flat, gallery.iterrows()):
    ax.imshow(preprocess_to_224(r.path, "none", None), cmap="gray")
    ax.imshow(cams[r.path], cmap="jet", alpha=0.35)
    ax.set_title(f"{r.outcome} | {r.subtype} | p={r.p:.2f}\n{r.file[:28]}", fontsize=8)
for ax in axes.flat:
    ax.axis("off")
plt.suptitle("Grad-CAM (ResNet-18, held-out fold model): most confident errors + correct examples")
plt.tight_layout(); plt.savefig(os.path.join(M5_ROOT, "F6_gradcam_gallery.png"), dpi=200); plt.show()

## 10. Summary and run manifest

In [ ]:
manifest = {
    "base_paper": "Kundu et al. (2021), PLOS ONE 16(9): e0256630",
    "dataset": {"name": "Kermany pediatric chest X-ray", "images": len(df), "classes": df["class"].value_counts().to_dict()},
    "protocol": {"folds": N_FOLDS, "seed": SEED, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LEARNING_RATE,
                 "optimizer": "Adam", "loss": "CrossEntropy", "scheduler": "ReduceLROnPlateau(0.1, patience=3)",
                 "internal_val": INTERNAL_VAL_FRACTION, "image_size": IMAGE_SIZE, "models": MODELS},
    "m4_variants": {k: {"kind": v[0], "clip_limit": v[1], "tile": list(CLAHE_TILE)} for k, v in VARIANTS.items()},
    "m3_predictions": M3_PREDICTIONS,
    "finished": {n: sorted(int(f) for f in p["fold"].unique()) for n, p in PRED.items()},
    "created": datetime.now().isoformat(),
}
with open(os.path.join(OUTPUT_ROOT, "experiment_manifest.json"), "w") as fh:
    json.dump(manifest, fh, indent=2)

print("Headline (Kundu ensemble, five-fold mean ± SD %):")
heads = ["accuracy", "precision", "recall", "f1", "auc_hard_label", "auc_probability", "Normal_recall"]
display(PER_FOLD[PER_FOLD.model == "ensemble"].groupby("experiment")[heads].agg(fmt))

### Where each result goes in the IEEE paper
| Paper section | Source in this notebook |
|---|---|
| III. Methods | §2 data, §3 method, §4 protocol, §5 checks |
| IV-A. Reproduction | §7.1 table, §7.2 confusion matrix + ROC, §7.3 matched comparison with Kundu et al. |
| IV-B. CLAHE (M4) | §8.1 E1 table + M3 prediction check, §8.2 ablation figure, §8.3 what moved |
| V. Failure analysis (M5) | §9.1 confident errors, §9.2 calibration/threshold, §9.3 subtypes, §9.4 ensemble behaviour, §9.5 patient leakage, §9.6 training dynamics, §9.7 Grad-CAM |
| VI. Limitations | One dataset, pediatric only, image-level folds (§9.5), no external test set, Grad-CAM only for ResNet-18, operating points chosen on the same data |

*Not a clinical tool:* this is a research reproduction on a public benchmark.